# WaterSoftHack  
## Cloud Computing Hands-On with NRP Nautilus  
### Setup and Familiarization Notebook

Welcome to the WaterSoftHack hands-on session.

In this workshop, we are **teaching cloud computing**, and we are using **NRP Nautilus** as the platform.

This notebook is therefore **not** a generic Python tutorial and **not** a deep Nautilus administration guide.  
It is a **setup and familiarization notebook** that helps you experience the cloud workflow through Nautilus.

---

## Learning goal

By the end of this notebook, you should be able to:

- recognize that your code is running on a **remote provisioned environment**
- identify the environment that Nautilus gave you
- inspect basic compute resources available in that environment
- understand where your files are being stored
- distinguish between **persistent** and **ephemeral** work habits
- create a clean workspace for the rest of the workshop
- save outputs in the right place

---

## What this notebook is about

We already covered cloud concepts in the presentations.

So in this notebook, we focus on the hands-on questions:

- Where is this notebook actually running?
- What machine am I using?
- What resources did I get?
- Where should I store my files?
- What happens if I disconnect?
- How do I use this environment correctly for the rest of the workshop?

# 1. Start with the big practical question

## Where does this notebook run?

This notebook runs on a **remote environment on NRP Nautilus**, not on your laptop.

Your browser is local, but the notebook kernel and Python execution happen remotely in the environment you spawned through Nautilus JupyterHub.

That is one of the cloud ideas we want you to experience directly:
- you access compute through a browser
- the compute is remote
- resources are provisioned for you
- your workflow depends on the storage and lifecycle of that remote environment

Run the next cell.

In [ ]:
print("WaterSoftHack cloud hands-on: connected to remote notebook environment.")

# 2. Inspect the remote environment

This cell helps answer:
- who am I in this environment?
- what host am I on?
- what is my home directory?
- where is my current working directory?

In [ ]:
import os
import sys
import platform

print("Python:", sys.version.splitlines()[0])
print("Hostname:", platform.node())
print("Platform:", platform.platform())
print("User:", os.environ.get("USER", "(unknown)"))
print("HOME:", os.environ.get("HOME", "(unknown)"))
print("PWD:", os.getcwd())

## Reflection

Notice that this information is describing the **remote environment**, not your own laptop.

That is exactly part of the cloud workflow:
you are interacting with a remote machine through an interface.

# 3. Inspect the notebook environment metadata

On Nautilus, the notebook environment depends on what was spawned for you.
That means it is useful to inspect environment variables that tell you something about the session.

In [ ]:
interesting = [
    "HOME", "PWD", "USER", "SHELL",
    "JUPYTERHUB_USER", "JUPYTERHUB_SERVER_NAME",
    "JUPYTERHUB_SERVICE_PREFIX", "JUPYTER_IMAGE_SPEC",
    "NB_USER", "NB_UID", "CUDA_VISIBLE_DEVICES"
]

for key in interesting:
    print(f"{key} = {os.environ.get(key, '(not set)')}")

# 4. What resources were provisioned for you?

Cloud platforms are useful because they provision resources remotely.

Run the next cells to inspect:
- CPU
- memory
- disk
- optionally GPU

In [ ]:
import os
print("os.cpu_count() =", os.cpu_count())

In [ ]:
!nproc || true
!free -h || true
!cat /proc/meminfo | head -n 10

In [ ]:
!df -h

## Reflection

This is one of the core cloud-computing experiences:
you did not configure this machine by hand on your laptop,
but you still received a working environment with real resources.

# 5. Check whether a GPU is available

Some cloud-style environments provide CPU only, while others expose GPUs.

Depending on how your Nautilus session was spawned, you may or may not see one.

In [ ]:
!which nvidia-smi || true
!nvidia-smi || true

In [ ]:
gpu_report = {}

try:
    import torch
    gpu_report["torch_installed"] = True
    gpu_report["torch_cuda_available"] = torch.cuda.is_available()
    gpu_report["torch_device_count"] = torch.cuda.device_count()
    if torch.cuda.is_available():
        gpu_report["torch_device_name_0"] = torch.cuda.get_device_name(0)
except Exception as e:
    gpu_report["torch"] = f"not available ({type(e).__name__})"

gpu_report

# 6. Inspect the filesystem

Cloud work is not just about compute.  
It is also about understanding **where your files live**.

Run the next cells to inspect the filesystem and your home directory.

In [ ]:
from pathlib import Path

home = Path.home()
print("Home directory:", home)
print()
print("Top-level entries in HOME:")
for p in sorted(home.iterdir()):
    print("-", p.name)

In [ ]:
!mount | head -n 40

In [ ]:
!du -sh $HOME 2>/dev/null || true
!df -h $HOME

## Reflection

At this point, you should start asking:

- which paths are safe for my workshop files?
- what looks persistent?
- what looks temporary?
- how much space do I actually have?

In a cloud setting, those questions matter.

# 7. Create your workshop workspace

A good cloud habit is to create a clearly named workspace and keep your work organized.

By default, we create the workshop folder in `HOME`.
If your instructors gave you a different persistent location, change it before running.

In [ ]:
from pathlib import Path

workspace_root = Path.home()
workspace = workspace_root / "watersofthack_cloud_nautilus"

workspace.mkdir(parents=True, exist_ok=True)

print("Workspace created at:")
print(workspace)

# 8. Write a test file into the workspace

This verifies that your chosen location is writable.

In [ ]:
notes_path = workspace / "cloud_notes.txt"
notes_path.write_text(
    "WaterSoftHack cloud computing hands-on\n"
    "Running on NRP Nautilus\n"
    "Workspace write test succeeded\n",
    encoding="utf-8"
)

print("Wrote:", notes_path)
print()
print(notes_path.read_text(encoding="utf-8"))

# 9. Save an environment snapshot

In cloud work, it is useful to record the environment you were given.
This helps with reproducibility and debugging.

In [ ]:
import json
import platform
import os
import sys

snapshot = {
    "python": sys.version,
    "platform": platform.platform(),
    "hostname": platform.node(),
    "user": os.environ.get("USER"),
    "home": os.environ.get("HOME"),
    "cwd": os.getcwd(),
    "cpu_count": os.cpu_count(),
    "cuda_visible_devices": os.environ.get("CUDA_VISIBLE_DEVICES"),
}

snapshot_path = workspace / "environment_snapshot.json"
snapshot_path.write_text(json.dumps(snapshot, indent=2), encoding="utf-8")

print("Saved:", snapshot_path)
print(snapshot_path.read_text(encoding="utf-8"))

# 10. Create and save a simple dataset

The goal here is not the data analysis itself.

The goal is to practice a cloud workflow:
- create something in the remote environment
- save it in your workspace
- reload it
- confirm the file exists where you expect

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "station_id": ["A", "B", "C", "D"],
    "flow_cms": [12.4, 18.1, 9.7, 22.3],
    "rain_mm": [0.0, 4.2, 0.8, 7.5]
})

csv_path = workspace / "cloud_test_dataset.csv"
df.to_csv(csv_path, index=False)

print("Saved:", csv_path)
df

In [ ]:
df2 = pd.read_csv(csv_path)
df2

# 11. Produce and save an output file

In a cloud notebook environment, outputs should be saved deliberately.

Run the next cell to save a figure into your workspace.

In [ ]:
import matplotlib.pyplot as plt

plot_path = workspace / "cloud_test_plot.png"

plt.figure(figsize=(6, 4))
plt.bar(df2["station_id"], df2["flow_cms"])
plt.title("WaterSoftHack cloud test plot")
plt.xlabel("Station")
plt.ylabel("Flow (m^3/s)")
plt.tight_layout()
plt.savefig(plot_path, dpi=150)
plt.show()

print("Saved:", plot_path)

# 12. Verify the workspace contents from the shell

A useful cloud habit is checking that your files are really where you think they are.

In [ ]:
!pwd
!ls -lah ~
!ls -lah ~/watersofthack_cloud_nautilus

# 13. Persistent vs ephemeral thinking

This is one of the most important practical lessons in remote/cloud environments.

## Good habit
Save notebooks, outputs, and important data into a location intended to persist.

## Bad habit
Assume that anything inside the container environment is permanent just because it exists right now.

Use the next text box as a short reflection prompt.

### Reflection prompt

Write one or two sentences in a new markdown cell answering:

- What files from today would you definitely want to store in persistent space?
- What kinds of things could be regenerated if the environment were restarted?

# 14. Optional package checks

The exact image on Nautilus determines which libraries are already installed.
Run this cell to inspect some common scientific packages.

In [ ]:
packages = [
    "numpy", "pandas", "matplotlib", "scipy", "sklearn",
    "xarray", "netCDF4", "rasterio", "geopandas",
    "torch", "tensorflow"
]

availability = {}
for pkg in packages:
    try:
        module = __import__(pkg)
        availability[pkg] = getattr(module, "__version__", "installed")
    except Exception as e:
        availability[pkg] = f"not available ({type(e).__name__})"

availability

# 15. Optional shell/tool checks

These checks help you see what else is available in the remote environment.

In [ ]:
!which git || true
!git --version || true
!which kubectl || true
!kubectl version --client || true

# 16. What this notebook was really teaching

This notebook used Nautilus to make several cloud-computing ideas concrete:

- remote execution
- provisioned resources
- browser-based access to compute
- shared infrastructure
- storage awareness
- persistence vs ephemerality
- reproducibility through saved environment details

So even though we used Nautilus specifically, the broader lesson is about working effectively with remote cloud-style environments.

# 17. Final checklist

You are done with this setup notebook if you have completed all of the following:

- [ ] confirmed that your notebook is running remotely on Nautilus
- [ ] inspected host, user, home directory, and working directory
- [ ] checked CPU, memory, and disk information
- [ ] checked whether GPU support is available
- [ ] created `~/watersofthack_cloud_nautilus`
- [ ] saved a text file there
- [ ] saved `environment_snapshot.json`
- [ ] saved a CSV there
- [ ] saved a plot there
- [ ] listed the workspace contents from the shell

If all of those are done, you are ready for the next WaterSoftHack notebook.

# Done

This notebook was designed as a **cloud-computing familiarization exercise using NRP Nautilus as the platform**.